In [3]:
from pathlib import Path
import pandas as pd

print("02.23 STEP 1 - INSPECT BASELINE FOR HIDDEN BUSINESS EVENTS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print()
print("FILES")
print("-" * 65)
print(f"Sales files found: {len(sales_files)}")

# ---------------------------------------------------------
# LOAD SALES FILES
# ---------------------------------------------------------

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)
    sales_list.append(df)

sales = pd.concat(
    sales_list,
    ignore_index=True
)

print()
print("BASELINE DATA")
print("-" * 65)

print(f"Total rows: {len(sales):,}")

# ---------------------------------------------------------
# SHOW ACTUAL COLUMNS
# ---------------------------------------------------------

print()
print("AVAILABLE COLUMNS")
print("-" * 65)

print(list(sales.columns))

# ---------------------------------------------------------
# DATE INSPECTION
# ---------------------------------------------------------

if "order_date" in sales.columns:

    sales["order_date_original"] = sales["order_date"]

    sales["order_date"] = pd.to_datetime(
        sales["order_date"],
        errors="coerce"
    )

    invalid_date_count = sales["order_date"].isna().sum()

    valid_dates = sales["order_date"].dropna()

    print()
    print("DATE VALIDATION")
    print("-" * 65)

    print(f"Valid dates: {len(valid_dates):,}")
    print(f"Invalid dates: {invalid_date_count:,}")

    if len(valid_dates) > 0:
        print(
            f"Valid date range: "
            f"{valid_dates.min().date()} "
            f"to "
            f"{valid_dates.max().date()}"
        )

# ---------------------------------------------------------
# MONTHLY TRANSACTION VOLUME
# ---------------------------------------------------------

if "order_date" in sales.columns:

    sales_valid_dates = sales[
        sales["order_date"].notna()
    ].copy()

    sales_valid_dates["year_month"] = (
        sales_valid_dates["order_date"]
        .dt.to_period("M")
        .astype(str)
    )

    monthly_volume = (
        sales_valid_dates
        .groupby("year_month")
        .size()
        .reset_index(name="transaction_count")
    )

    print()
    print("MONTHLY TRANSACTION VOLUME")
    print("-" * 65)

    print(
        monthly_volume.to_string(index=False)
    )

# ---------------------------------------------------------
# PRODUCT INSPECTION
# ---------------------------------------------------------

if "product_id" in sales.columns:

    print()
    print("PRODUCT INFORMATION")
    print("-" * 65)

    print(
        f"Unique product IDs: "
        f"{sales['product_id'].nunique():,}"
    )

# ---------------------------------------------------------
# STORE INSPECTION
# ---------------------------------------------------------

if "store_id" in sales.columns:

    print()
    print("STORE INFORMATION")
    print("-" * 65)

    print(
        f"Unique stores: "
        f"{sales['store_id'].nunique():,}"
    )

# ---------------------------------------------------------
# CUSTOMER INSPECTION
# ---------------------------------------------------------

if "customer_id" in sales.columns:

    print()
    print("CUSTOMER INFORMATION")
    print("-" * 65)

    print(
        f"Unique customer IDs: "
        f"{sales['customer_id'].nunique():,}"
    )

# ---------------------------------------------------------
# PAYMENT METHOD
# ---------------------------------------------------------

if "payment_method" in sales.columns:

    print()
    print("PAYMENT METHODS")
    print("-" * 65)

    print(
        sales["payment_method"]
        .value_counts(dropna=False)
        .to_string()
    )

# ---------------------------------------------------------
# CHANNEL
# ---------------------------------------------------------

if "channel" in sales.columns:

    print()
    print("SALES CHANNELS")
    print("-" * 65)

    print(
        sales["channel"]
        .value_counts(dropna=False)
        .to_string()
    )

else:

    print()
    print("SALES CHANNELS")
    print("-" * 65)

    print("Channel column is not present in the current 02.22 output.")
    print("Skipped channel analysis.")

# ---------------------------------------------------------
# FINAL
# ---------------------------------------------------------

print()
print("=" * 65)
print("02.23 Step 1 inspection: COMPLETED")
print("=" * 65)

print()
print("Baseline inspection completed without modifying source data.")

02.23 STEP 1 - INSPECT BASELINE FOR HIDDEN BUSINESS EVENTS

FILES
-----------------------------------------------------------------
Sales files found: 32

BASELINE DATA
-----------------------------------------------------------------
Total rows: 2,118,200

AVAILABLE COLUMNS
-----------------------------------------------------------------
['sales_line_id', 'order_id', 'order_date', 'order_time', 'customer_id', 'product_id', 'store_id', 'salesperson_id', 'promotion_id', 'sales_channel', 'quantity', 'unit_selling_price', 'discount_pct', 'payment_method', 'order_status']

DATE VALIDATION
-----------------------------------------------------------------
Valid dates: 2,116,922
Invalid dates: 1,278
Valid date range: 2024-01-01 to 2099-12-31

MONTHLY TRANSACTION VOLUME
-----------------------------------------------------------------
year_month  transaction_count
   2024-01              53152
   2024-02              50351
   2024-03              55955
   2024-04              57052
   2024-05

In [4]:
print("02.23 STEP 2 - DEFINE HIDDEN BUSINESS EVENT SCENARIOS")
print("=" * 65)

# ---------------------------------------------------------
# APPROVED HIDDEN BUSINESS EVENT SCENARIOS
# ---------------------------------------------------------

HIDDEN_EVENTS = {

    "regional_sales_shift": {
        "description": "Controlled change in sales contribution across selected regions",
        "target": "store_id / region",
        "type": "sales_volume"
    },

    "product_demand_shift": {
        "description": "Controlled increase or decrease in demand for selected products/categories",
        "target": "product_id / category",
        "type": "product_demand"
    },

    "channel_shift": {
        "description": "Controlled change in sales contribution by sales channel",
        "target": "sales_channel",
        "type": "channel_behavior"
    },

    "promotion_effect": {
        "description": "Controlled change in sales activity during selected promotion periods",
        "target": "promotion_id",
        "type": "promotion_behavior"
    },

    "seasonal_event": {
        "description": "Controlled temporary sales pattern during selected months",
        "target": "order_date",
        "type": "seasonality"
    }
}

# ---------------------------------------------------------
# DISPLAY SCENARIOS
# ---------------------------------------------------------

print()
print("HIDDEN BUSINESS EVENT SCENARIOS")
print("-" * 65)

for event_name, details in HIDDEN_EVENTS.items():

    print()
    print(f"Event: {event_name}")
    print(f"Description: {details['description']}")
    print(f"Target: {details['target']}")
    print(f"Type: {details['type']}")

# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

required_events = {
    "regional_sales_shift",
    "product_demand_shift",
    "channel_shift",
    "promotion_effect",
    "seasonal_event"
}

defined_events = set(HIDDEN_EVENTS.keys())

events_valid = required_events == defined_events

print()
print("VALIDATION")
print("-" * 65)

print(f"Required events: {len(required_events)}")
print(f"Defined events: {len(defined_events)}")
print(f"All required events defined: {events_valid}")

if events_valid:
    print()
    print("02.23 Step 2 validation: PASSED")
else:
    print()
    print("02.23 Step 2 validation: FAILED")

02.23 STEP 2 - DEFINE HIDDEN BUSINESS EVENT SCENARIOS

HIDDEN BUSINESS EVENT SCENARIOS
-----------------------------------------------------------------

Event: regional_sales_shift
Description: Controlled change in sales contribution across selected regions
Target: store_id / region
Type: sales_volume

Event: product_demand_shift
Description: Controlled increase or decrease in demand for selected products/categories
Target: product_id / category
Type: product_demand

Event: channel_shift
Description: Controlled change in sales contribution by sales channel
Target: sales_channel
Type: channel_behavior

Event: promotion_effect
Description: Controlled change in sales activity during selected promotion periods
Target: promotion_id
Type: promotion_behavior

Event: seasonal_event
Description: Controlled temporary sales pattern during selected months
Target: order_date
Type: seasonality

VALIDATION
-----------------------------------------------------------------
Required events: 5
Defined e

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.23 STEP 3 - INJECT CONTROLLED HIDDEN BUSINESS EVENTS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "hidden_business_events_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

print()
print(f"Input sales files: {len(sales_files)}")

# ---------------------------------------------------------
# CONTROLLED EVENT SETTINGS
# ---------------------------------------------------------

RNG = np.random.default_rng(2026)

EVENT_RATE = 0.005

total_input_rows = 0
total_output_rows = 0

event_counts = {
    "regional_sales_shift": 0,
    "product_demand_shift": 0,
    "channel_shift": 0,
    "promotion_effect": 0,
    "seasonal_event": 0
}

# ---------------------------------------------------------
# PROCESS EACH SALES FILE
# ---------------------------------------------------------

for file in sales_files:

    sales = pd.read_csv(file)

    total_input_rows += len(sales)

    # Preserve original row structure
    adjusted = sales.copy()

    # -----------------------------------------------------
    # 1. REGIONAL SALES SHIFT
    # -----------------------------------------------------
    # Use store_id because region is not directly present
    # in the sales transaction file.

    if "store_id" in adjusted.columns:

        mask = (
            RNG.random(len(adjusted)) < EVENT_RATE
        )

        event_counts["regional_sales_shift"] += int(mask.sum())

        # Controlled quantity increase
        if "quantity" in adjusted.columns:

            adjusted.loc[mask, "quantity"] = (
                adjusted.loc[mask, "quantity"]
                .astype(float)
                * 1.25
            ).round().clip(lower=1)

    # -----------------------------------------------------
    # 2. PRODUCT DEMAND SHIFT
    # -----------------------------------------------------

    if "product_id" in adjusted.columns:

        mask = (
            RNG.random(len(adjusted)) < EVENT_RATE
        )

        event_counts["product_demand_shift"] += int(mask.sum())

        if "quantity" in adjusted.columns:

            adjusted.loc[mask, "quantity"] = (
                adjusted.loc[mask, "quantity"]
                .astype(float)
                * 1.20
            ).round().clip(lower=1)

    # -----------------------------------------------------
    # 3. CHANNEL SHIFT
    # -----------------------------------------------------

    if "sales_channel" in adjusted.columns:

        mask = (
            RNG.random(len(adjusted)) < EVENT_RATE
        )

        event_counts["channel_shift"] += int(mask.sum())

        available_channels = (
            adjusted.loc[
                adjusted["sales_channel"].notna(),
                "sales_channel"
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        if len(available_channels) > 1:

            replacement_channel = available_channels[0]

            adjusted.loc[
                mask,
                "sales_channel"
            ] = replacement_channel

    # -----------------------------------------------------
    # 4. PROMOTION EFFECT
    # -----------------------------------------------------

    if "promotion_id" in adjusted.columns:

        mask = (
            RNG.random(len(adjusted)) < EVENT_RATE
        )

        event_counts["promotion_effect"] += int(mask.sum())

        if "discount_pct" in adjusted.columns:

            adjusted.loc[mask, "discount_pct"] = (
                adjusted.loc[mask, "discount_pct"]
                .astype(float)
                + 5
            ).clip(upper=95)

    # -----------------------------------------------------
    # 5. SEASONAL EVENT
    # -----------------------------------------------------

    if "order_date" in adjusted.columns:

        parsed_dates = pd.to_datetime(
            adjusted["order_date"],
            errors="coerce"
        )

        seasonal_mask = (
            parsed_dates.dt.month.isin([10, 11, 12])
            & (RNG.random(len(adjusted)) < EVENT_RATE)
        )

        event_counts["seasonal_event"] += int(
            seasonal_mask.sum()
        )

        if "quantity" in adjusted.columns:

            adjusted.loc[
                seasonal_mask,
                "quantity"
            ] = (
                adjusted.loc[
                    seasonal_mask,
                    "quantity"
                ]
                .astype(float)
                * 1.30
            ).round().clip(lower=1)

    # -----------------------------------------------------
    # SAVE
    # -----------------------------------------------------

    output_file = OUTPUT_DIR / file.name

    adjusted.to_csv(
        output_file,
        index=False
    )

    total_output_rows += len(adjusted)


# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------

print()
print("HIDDEN EVENT INJECTION SUMMARY")
print("-" * 65)

for event_name, count in event_counts.items():

    print(
        f"{event_name}: {count:,} rows"
    )

print()
print("FINAL VALIDATION")
print("-" * 65)

print(
    f"Input rows:  {total_input_rows:,}"
)

print(
    f"Output rows: {total_output_rows:,}"
)

print(
    f"Input files:  {len(sales_files)}"
)

print(
    f"Output files: {len(list(OUTPUT_DIR.glob('sales_*.csv')))}"
)

row_count_valid = (
    total_input_rows == total_output_rows
)

file_count_valid = (
    len(sales_files)
    == len(list(OUTPUT_DIR.glob("sales_*.csv")))
)

print(
    f"Row count preserved: {row_count_valid}"
)

print(
    f"File count preserved: {file_count_valid}"
)

if row_count_valid and file_count_valid:
    print()
    print("02.23 Step 3 injection: PASSED")
else:
    print()
    print("02.23 Step 3 injection: FAILED")

02.23 STEP 3 - INJECT CONTROLLED HIDDEN BUSINESS EVENTS

Input sales files: 32

HIDDEN EVENT INJECTION SUMMARY
-----------------------------------------------------------------
regional_sales_shift: 10,603 rows
product_demand_shift: 10,466 rows
channel_shift: 10,668 rows
promotion_effect: 10,516 rows
seasonal_event: 2,508 rows

FINAL VALIDATION
-----------------------------------------------------------------
Input rows:  2,118,200
Output rows: 2,118,200
Input files:  32
Output files: 32
Row count preserved: True
File count preserved: True

02.23 Step 3 injection: PASSED


In [6]:
from pathlib import Path
import pandas as pd

print("02.23 STEP 4 - VALIDATE HIDDEN BUSINESS EVENTS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "inconsistent_text_category_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "hidden_business_events_adjusted"
    / "sales_adjusted"
)

input_files = sorted(INPUT_DIR.glob("sales_*.csv"))
output_files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

print()
print("FILE VALIDATION")
print("-" * 65)

print(f"Input files:  {len(input_files)}")
print(f"Output files: {len(output_files)}")

file_count_valid = (
    len(input_files) == 32
    and len(output_files) == 32
)

# ---------------------------------------------------------
# VALIDATE FILE-BY-FILE
# ---------------------------------------------------------

total_input_rows = 0
total_output_rows = 0

total_quantity_changes = 0
total_discount_changes = 0
total_channel_changes = 0

files_with_quantity_changes = 0
files_with_discount_changes = 0
files_with_channel_changes = 0

schema_valid = True

for input_file in input_files:

    output_file = OUTPUT_DIR / input_file.name

    if not output_file.exists():
        print(f"Missing output file: {input_file.name}")
        continue

    original = pd.read_csv(input_file)
    adjusted = pd.read_csv(output_file)

    total_input_rows += len(original)
    total_output_rows += len(adjusted)

    # -----------------------------------------------------
    # SCHEMA VALIDATION
    # -----------------------------------------------------

    if list(original.columns) != list(adjusted.columns):
        schema_valid = False
        print(
            f"Schema mismatch: {input_file.name}"
        )

    # -----------------------------------------------------
    # QUANTITY CHANGES
    # -----------------------------------------------------

    if "quantity" in original.columns:

        quantity_changed = (
            original["quantity"].astype(str)
            != adjusted["quantity"].astype(str)
        )

        quantity_count = int(quantity_changed.sum())

        total_quantity_changes += quantity_count

        if quantity_count > 0:
            files_with_quantity_changes += 1

    # -----------------------------------------------------
    # DISCOUNT CHANGES
    # -----------------------------------------------------

    if "discount_pct" in original.columns:

        original_discount = pd.to_numeric(
            original["discount_pct"],
            errors="coerce"
        )

        adjusted_discount = pd.to_numeric(
            adjusted["discount_pct"],
            errors="coerce"
        )

        discount_changed = (
            original_discount.fillna(-999999)
            != adjusted_discount.fillna(-999999)
        )

        discount_count = int(discount_changed.sum())

        total_discount_changes += discount_count

        if discount_count > 0:
            files_with_discount_changes += 1

    # -----------------------------------------------------
    # SALES CHANNEL CHANGES
    # -----------------------------------------------------

    if "sales_channel" in original.columns:

        channel_changed = (
            original["sales_channel"].astype(str)
            != adjusted["sales_channel"].astype(str)
        )

        channel_count = int(channel_changed.sum())

        total_channel_changes += channel_count

        if channel_count > 0:
            files_with_channel_changes += 1


# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------

print()
print("ROW VALIDATION")
print("-" * 65)

print(
    f"Input rows:  {total_input_rows:,}"
)

print(
    f"Output rows: {total_output_rows:,}"
)

row_count_valid = (
    total_input_rows == total_output_rows
    and total_input_rows == 2_118_200
)

print(
    f"Row count preserved: {row_count_valid}"
)

# ---------------------------------------------------------
# CHANGE VALIDATION
# ---------------------------------------------------------

print()
print("HIDDEN EVENT CHANGE VALIDATION")
print("-" * 65)

print(
    f"Quantity changes: {total_quantity_changes:,}"
)

print(
    f"Discount changes: {total_discount_changes:,}"
)

print(
    f"Sales-channel changes: {total_channel_changes:,}"
)

print()
print("Files containing changes")
print("-" * 65)

print(
    f"Files with quantity changes: "
    f"{files_with_quantity_changes}"
)

print(
    f"Files with discount changes: "
    f"{files_with_discount_changes}"
)

print(
    f"Files with channel changes: "
    f"{files_with_channel_changes}"
)

# ---------------------------------------------------------
# VALIDATION LOGIC
# ---------------------------------------------------------

quantity_event_valid = (
    total_quantity_changes > 0
)

discount_event_valid = (
    total_discount_changes > 0
)

channel_event_valid = (
    total_channel_changes > 0
)

all_events_valid = (
    quantity_event_valid
    and discount_event_valid
    and channel_event_valid
)

# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------

print()
print("FINAL VALIDATION")
print("-" * 65)

print(
    f"File count preserved: {file_count_valid}"
)

print(
    f"Row count preserved: {row_count_valid}"
)

print(
    f"Schema preserved: {schema_valid}"
)

print(
    f"Quantity hidden pattern detected: "
    f"{quantity_event_valid}"
)

print(
    f"Discount hidden pattern detected: "
    f"{discount_event_valid}"
)

print(
    f"Channel hidden pattern detected: "
    f"{channel_event_valid}"
)

if (
    file_count_valid
    and row_count_valid
    and schema_valid
    and all_events_valid
):
    print()
    print("02.23 Step 4 validation: PASSED")
else:
    print()
    print("02.23 Step 4 validation: NEEDS REVIEW")

02.23 STEP 4 - VALIDATE HIDDEN BUSINESS EVENTS

FILE VALIDATION
-----------------------------------------------------------------
Input files:  32
Output files: 32

ROW VALIDATION
-----------------------------------------------------------------
Input rows:  2,118,200
Output rows: 2,118,200
Row count preserved: True

HIDDEN EVENT CHANGE VALIDATION
-----------------------------------------------------------------
Quantity changes: 4,058
Discount changes: 10,515
Sales-channel changes: 3,760

Files containing changes
-----------------------------------------------------------------
Files with quantity changes: 32
Files with discount changes: 32
Files with channel changes: 32

FINAL VALIDATION
-----------------------------------------------------------------
File count preserved: True
Row count preserved: True
Schema preserved: True
Quantity hidden pattern detected: True
Discount hidden pattern detected: True
Channel hidden pattern detected: True

02.23 Step 4 validation: PASSED


In [8]:
from pathlib import Path
import pandas as pd

print("02.23 STEP 5 - FINAL HIDDEN BUSINESS EVENT VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "hidden_business_events_adjusted"
    / "sales_adjusted"
)

files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

total_rows = 0
total_duplicates = 0
schema_reference = None
schema_consistent = True

print()
print("OUTPUT VALIDATION")
print("-" * 65)

print(f"Files found: {len(files)}")

for file in files:

    df = pd.read_csv(file)

    total_rows += len(df)

    total_duplicates += int(
        df.duplicated().sum()
    )

    current_schema = list(df.columns)

    if schema_reference is None:
        schema_reference = current_schema
    elif current_schema != schema_reference:
        schema_consistent = False

print(f"Total rows: {total_rows:,}")
print(f"Duplicate complete rows: {total_duplicates:,}")

print()
print("EXPECTED VS ACTUAL")
print("-" * 65)

print(f"Expected files: {EXPECTED_FILES}")
print(f"Actual files:   {len(files)}")

print(f"Expected rows: {EXPECTED_ROWS:,}")
print(f"Actual rows:   {total_rows:,}")

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

file_count_valid = (
    len(files) == EXPECTED_FILES
)

row_count_valid = (
    total_rows == EXPECTED_ROWS
)

duplicate_check = (
    total_duplicates == 0
)

# IMPORTANT:
# Actual sales column is "channel", not "sales_channel"

required_columns = {
    "sales_line_id",
    "order_id",
    "order_date",
    "order_time",
    "customer_id",
    "product_id",
    "store_id",
    "salesperson_id",
    "promotion_id",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "channel",
    "payment_method",
    "status"
}

columns_valid = (
    schema_reference is not None
    and required_columns.issubset(
        set(schema_reference)
    )
)

print()
print("FINAL VALIDATION")
print("-" * 65)

print(f"File count valid: {file_count_valid}")
print(f"Row count valid: {row_count_valid}")
print(f"Schema consistent: {schema_consistent}")
print(f"Required columns present: {columns_valid}")
print(f"No duplicate complete rows: {duplicate_check}")

# ---------------------------------------------------------
# SAVE VALIDATION SUMMARY
# ---------------------------------------------------------

validation_summary = pd.DataFrame([
    {
        "validation": "file_count",
        "expected": EXPECTED_FILES,
        "actual": len(files),
        "passed": file_count_valid
    },
    {
        "validation": "row_count",
        "expected": EXPECTED_ROWS,
        "actual": total_rows,
        "passed": row_count_valid
    },
    {
        "validation": "schema_consistent",
        "expected": True,
        "actual": schema_consistent,
        "passed": schema_consistent
    },
    {
        "validation": "required_columns",
        "expected": True,
        "actual": columns_valid,
        "passed": columns_valid
    },
    {
        "validation": "duplicate_complete_rows",
        "expected": 0,
        "actual": total_duplicates,
        "passed": duplicate_check
    }
])

summary_file = (
    OUTPUT_DIR
    / "hidden_business_events_validation_summary.csv"
)

validation_summary.to_csv(
    summary_file,
    index=False
)

# ---------------------------------------------------------
# FINAL RESULT
# ---------------------------------------------------------

all_valid = (
    file_count_valid
    and row_count_valid
    and schema_consistent
    and columns_valid
    and duplicate_check
)

print()
print("Validation summary saved to:")
print(summary_file)

print()

if all_valid:
    print("02.23 Step 5 validation: PASSED")
    print("02.23 hidden business events: COMPLETED")
else:
    print("02.23 Step 5 validation: NEEDS REVIEW")

02.23 STEP 5 - FINAL HIDDEN BUSINESS EVENT VALIDATION

OUTPUT VALIDATION
-----------------------------------------------------------------
Files found: 32
Total rows: 2,118,200
Duplicate complete rows: 0

EXPECTED VS ACTUAL
-----------------------------------------------------------------
Expected files: 32
Actual files:   32
Expected rows: 2,118,200
Actual rows:   2,118,200

FINAL VALIDATION
-----------------------------------------------------------------
File count valid: True
Row count valid: True
Schema consistent: True
Required columns present: False
No duplicate complete rows: True

Validation summary saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\hidden_business_events_adjusted\sales_adjusted\hidden_business_events_validation_summary.csv

02.23 Step 5 validation: NEEDS REVIEW


In [9]:
from pathlib import Path
import pandas as pd

print("02.23 STEP 5 - CHECK ACTUAL OUTPUT COLUMNS")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "hidden_business_events_adjusted"
    / "sales_adjusted"
)

files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

print(f"\nFiles found: {len(files)}")

# Read first output file
sample_file = files[0]

sample_df = pd.read_csv(sample_file)

print("\nFirst output file:")
print(sample_file.name)

print("\nACTUAL COLUMNS IN OUTPUT FILE")
print("-" * 65)

for i, column in enumerate(sample_df.columns, start=1):
    print(f"{i:2}. {column}")

print("\nCOLUMN COUNT")
print("-" * 65)
print(f"Total columns: {len(sample_df.columns)}")

print("\n02.23 Step 5 column inspection: COMPLETED")


02.23 STEP 5 - CHECK ACTUAL OUTPUT COLUMNS

Files found: 32

First output file:
sales_2024-01.csv

ACTUAL COLUMNS IN OUTPUT FILE
-----------------------------------------------------------------
 1. sales_line_id
 2. order_id
 3. order_date
 4. order_time
 5. customer_id
 6. product_id
 7. store_id
 8. salesperson_id
 9. promotion_id
10. sales_channel
11. quantity
12. unit_selling_price
13. discount_pct
14. payment_method
15. order_status

COLUMN COUNT
-----------------------------------------------------------------
Total columns: 15

02.23 Step 5 column inspection: COMPLETED


In [10]:
from pathlib import Path
import pandas as pd

print("02.23 STEP 5 - FINAL HIDDEN BUSINESS EVENT VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "hidden_business_events_adjusted"
    / "sales_adjusted"
)

files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

EXPECTED_FILE_COUNT = 32
EXPECTED_ROW_COUNT = 2_118_200

REQUIRED_COLUMNS = {
    "sales_line_id",
    "order_id",
    "order_date",
    "order_time",
    "customer_id",
    "product_id",
    "store_id",
    "salesperson_id",
    "promotion_id",
    "sales_channel",
    "quantity",
    "unit_selling_price",
    "discount_pct",
    "payment_method",
    "order_status"
}

total_rows = 0
schema_consistent = True
required_columns_present = True
duplicate_complete_rows = 0

reference_columns = None

for file in files:

    df = pd.read_csv(file)

    total_rows += len(df)

    current_columns = set(df.columns)

    # Check schema consistency
    if reference_columns is None:
        reference_columns = list(df.columns)
    elif list(df.columns) != reference_columns:
        schema_consistent = False

    # Check required columns
    missing_columns = REQUIRED_COLUMNS - current_columns

    if missing_columns:
        required_columns_present = False
        print(f"\nMissing columns in {file.name}:")
        print(sorted(missing_columns))

    # Check duplicate complete rows
    duplicate_complete_rows += df.duplicated().sum()


print("\nOUTPUT VALIDATION")
print("-" * 65)

print(f"Files found: {len(files)}")
print(f"Total rows: {total_rows}")
print(f"Expected files: {EXPECTED_FILE_COUNT}")
print(f"Expected rows: {EXPECTED_ROW_COUNT}")

print("\nSCHEMA VALIDATION")
print("-" * 65)

print(f"Schema consistent: {schema_consistent}")
print(f"Required columns present: {required_columns_present}")

print("\nDUPLICATE VALIDATION")
print("-" * 65)

print(f"Duplicate complete rows: {duplicate_complete_rows}")

# Final validation
file_count_valid = len(files) == EXPECTED_FILE_COUNT
row_count_valid = total_rows == EXPECTED_ROW_COUNT
duplicate_rows_valid = duplicate_complete_rows == 0

final_validation = (
    file_count_valid
    and row_count_valid
    and schema_consistent
    and required_columns_present
    and duplicate_rows_valid
)

print("\nFINAL VALIDATION")
print("-" * 65)

print(f"File count valid: {file_count_valid}")
print(f"Row count valid: {row_count_valid}")
print(f"Schema consistent: {schema_consistent}")
print(f"Required columns present: {required_columns_present}")
print(f"No duplicate complete rows: {duplicate_rows_valid}")

if final_validation:
    print("\n02.23 Step 5 validation: PASSED")
else:
    print("\n02.23 Step 5 validation: NEEDS REVIEW")
    

02.23 STEP 5 - FINAL HIDDEN BUSINESS EVENT VALIDATION

OUTPUT VALIDATION
-----------------------------------------------------------------
Files found: 32
Total rows: 2118200
Expected files: 32
Expected rows: 2118200

SCHEMA VALIDATION
-----------------------------------------------------------------
Schema consistent: True
Required columns present: True

DUPLICATE VALIDATION
-----------------------------------------------------------------
Duplicate complete rows: 0

FINAL VALIDATION
-----------------------------------------------------------------
File count valid: True
Row count valid: True
Schema consistent: True
Required columns present: True
No duplicate complete rows: True

02.23 Step 5 validation: PASSED
